Dependencies

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [ ]:
import torch
import numpy as np
import random

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import (
    LoraConfig,
    IA3Config,
    PrefixTuningConfig,
    get_peft_model
)
import evaluate


In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


Model

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def preprocess(examples):
    return tokenizer(
        examples["sentence1"],
        examples["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


Dataset (MRPC)

In [ ]:
task = "stsb"
dataset = load_dataset("glue", task)
metric = evaluate.load("glue", "stsb")
num_labels = 1


README.md: 0.00B [00:00, ?B/s]

stsb/train-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

stsb/validation-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

stsb/test-00000-of-00001.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

Dataset helper

In [ ]:
def get_subset(dataset, fraction, seed=42):
    size = int(len(dataset) * fraction)
    return dataset.shuffle(seed=seed).select(range(size))

def build_datasets(fraction):
    small_train = get_subset(dataset["train"], fraction)

    train_ds = small_train.map(preprocess, batched=True)
    val_ds = dataset["validation"].map(preprocess, batched=True)

    keep_cols = ["input_ids", "attention_mask", "token_type_ids", "label"]
    train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols])
    val_ds = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_cols])

    train_ds = train_ds.rename_column("label", "labels")
    val_ds = val_ds.rename_column("label", "labels")

    train_ds.set_format("torch")
    val_ds.set_format("torch")

    return train_ds, val_ds

Metrices

In [ ]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze()
    return metric.compute(predictions=preds, references=labels)


Trainable param

In [ ]:
def print_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")

**Training arguments**

In [ ]:
training_args_full = TrainingArguments(
    output_dir="./outputs",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)

training_args = TrainingArguments(
    output_dir="./outputs",
    eval_strategy="epoch",
    learning_rate=1e-4,            # ↑ CRITICAL
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,            # ↑ more epochs
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none"
)



Train

In [ ]:
def train_and_eval(model, name, train_ds, val_ds):
    print(f"\n===== Training: {name} =====")
    print_trainable_params(model)

    args = training_args_full if name == "Full FT" else training_args_peft

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()
    return trainer.evaluate()


Model

In [ ]:
def build_models():
    # ===== Full Fine-Tuning =====
    model_full = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1,
        problem_type="regression"
    )

    # ===== BitFit =====
    model_bitfit = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1,
        problem_type="regression"
    )
    for name, param in model_bitfit.named_parameters():
        if "bias" not in name:
            param.requires_grad = False

    # ===== LoRA =====
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["query", "value"],
        lora_dropout=0.1,
        task_type="SEQ_CLS"
    )

    base_lora = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1,
        problem_type="regression"
    )
    model_lora = get_peft_model(base_lora, lora_config)

    # ===== IA³ =====
    ia3_config = IA3Config(task_type="SEQ_CLS")
    base_ia3 = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1,
        problem_type="regression"
    )
    model_ia3 = get_peft_model(base_ia3, ia3_config)

    # ===== Prefix Tuning =====
    prefix_config = PrefixTuningConfig(
        task_type="SEQ_CLS",
        num_virtual_tokens=30   # better for STS-B
    )
    base_prefix = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1,
        problem_type="regression"
    )
    model_prefix = get_peft_model(base_prefix, prefix_config)

    return model_full, model_bitfit, model_lora, model_ia3, model_prefix


Dataset fraction

In [ ]:
training_args_peft = training_args

fractions = [0.3, 0.5, 0.7, 1.0]
all_results = {}

for frac in fractions:
    print(f"\n===== Training with {int(frac*100)}% data =====")

    train_ds, val_ds = build_datasets(frac)
    model_full, model_bitfit, model_lora, model_ia3, model_prefix = build_models()

    results_frac = {}
    results_frac["Full FT"] = train_and_eval(model_full, "Full FT", train_ds, val_ds)
    results_frac["BitFit"]  = train_and_eval(model_bitfit, "BitFit", train_ds, val_ds)
    results_frac["LoRA"]    = train_and_eval(model_lora, "LoRA", train_ds, val_ds)
    results_frac["IA3"]     = train_and_eval(model_ia3, "IA3", train_ds, val_ds)
    results_frac["Prefix"] = train_and_eval(model_prefix, "Prefix", train_ds, val_ds)

    all_results[int(frac*100)] = results_frac

    del model_full, model_bitfit, model_lora, model_ia3, model_prefix
    torch.cuda.empty_cache()



===== Training with 30% data =====


Map:   0%|          | 0/1724 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Pearson,Spearmanr
1,1.928157,1.428210,0.738686,0.713750
2,1.233376,0.926330,0.797195,0.788053
3,1.001331,0.851764,0.804552,0.786835
4,1.034346,0.847358,0.806358,0.783948
5,0.948955,0.739880,0.825740,0.823625
6,0.928220,0.718580,0.829583,0.829850
7,0.827183,0.751133,0.825417,0.820431
8,0.863204,0.790232,0.820321,0.803452
9,0.835142,0.725738,0.830222,0.828201
10,0.862468,0.698189,0.834228,0.835264



===== Training: LoRA =====
Trainable params: 590,593 / 110,073,602 (0.5365%)


Epoch,Training Loss,Validation Loss,Pearson,Spearmanr
1,0.713354,0.561958,0.872344,0.868830
2,0.582506,0.509125,0.881360,0.876431
3,0.540389,0.547437,0.884484,0.882284
4,0.502530,0.508345,0.889580,0.886447
5,0.457498,0.512362,0.889574,0.886410
6,0.449046,0.492808,0.889321,0.885444
7,0.387591,0.493576,0.889615,0.885335
8,0.381878,0.528619,0.889620,0.886816
9,0.351820,0.492188,0.889373,0.884749
10,0.330351,0.503793,0.889576,0.886727



===== Training: IA3 =====
Trainable params: 65,281 / 109,548,290 (0.0596%)


Epoch,Training Loss,Validation Loss,Pearson,Spearmanr
1,1.591032,1.463005,0.612247,0.700077
2,1.311151,1.142615,0.707488,0.722616
3,1.177801,1.045703,0.746370,0.748711
4,1.179350,1.010212,0.768960,0.768107
5,1.084080,0.889660,0.786391,0.783989
6,1.023459,0.845783,0.797165,0.795199
7,0.943083,0.837358,0.804601,0.802644
8,1.032085,0.815065,0.809957,0.809706
9,0.943291,0.792058,0.814376,0.814425
10,0.901365,0.787240,0.817336,0.817651
